In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np

In [ ]:
from idinn.sourcing_model import SingleSourcingModel
from idinn.single_controller import BaseStockController
from idinn.demand import UniformDemand

single_sourcing_model = SingleSourcingModel(
    lead_time=2,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
)
controller_base = BaseStockController()
# z_star should be 11
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 29
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

In [ ]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
 )
controller_base = BaseStockController()
# z_star should be 4
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 10
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

In [ ]:
from idinn.sourcing_model import SingleSourcingModel
from idinn.single_controller import SingleSourcingNeuralController
from idinn.demand import UniformDemand
from torch.utils.tensorboard import SummaryWriter

single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=1, high=4),
)

controller_neural = SingleSourcingNeuralController()
controller_neural.fit(
    sourcing_model=single_sourcing_model,
    sourcing_periods=50,
    validation_sourcing_periods=1000,
    epochs=2000,
    tensorboard_writer=SummaryWriter(comment="_single_1"),
    seed=1,
)
controller_neural.get_average_cost(single_sourcing_model, sourcing_periods=1000)

In [ ]:
from idinn.sourcing_model import DualSourcingModel
from idinn.dual_controller import CappedDualIndexController
from idinn.demand import UniformDemand

dual_sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)
controller_cdi = CappedDualIndexController()
controller_cdi.fit(
    dual_sourcing_model,
    sourcing_periods=100
)
controller_cdi.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

In [ ]:
controller_cdi.predict(
    current_inventory=3, past_regular_orders=np.array([[1, 1, 1]]), past_expedited_orders=np.array([[0, 0, 0]])
)

In [2]:
from idinn.sourcing_model import DualSourcingModel
from idinn.dual_controller import DynamicProgrammingController
from idinn.demand import UniformDemand

dual_sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)
controller_dp = DynamicProgrammingController()
controller_dp.fit(
    dual_sourcing_model,
    max_iterations=10000,
    tolerance=1e-6
)
controller_dp.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

tensor(24.2900, grad_fn=<DivBackward0>)

In [3]:
import torch
from idinn.sourcing_model import DualSourcingModel
from idinn.dual_controller import DualSourcingNeuralController
from idinn.demand import UniformDemand
from torch.utils.tensorboard import SummaryWriter

dual_sourcing_model = DualSourcingModel(
    regular_lead_time=2,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    batch_size=256,
    init_inventory=6,
    demand_generator=UniformDemand(low=1, high=4),
)
controller_neural = DualSourcingNeuralController(
    hidden_layers=[128, 64, 32, 16, 8, 4],
    activation=torch.nn.CELU(alpha=1)
)
controller_neural.fit(
    sourcing_model=dual_sourcing_model,
    sourcing_periods=100,
    validation_sourcing_periods=1000,
    epochs=2000,
    tensorboard_writer=SummaryWriter(comment="dual"),
    seed=4,
)
controller_neural.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

tensor(17.9469, grad_fn=<DivBackward0>)

In [ ]:
from idinn.demand import CustomDemand

sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=CustomDemand({5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1})
)

In [ ]:
controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=1000000,
    tolerance=1e-6
)

In [ ]:
max_demand = 4
min_demand = 0
support = max_demand - min_demand
demand_prob = dict(
    zip(
        np.arange(min_demand, max_demand + 1),
        np.repeat(1.0 / (support + 1.0), int(support + 1)),
    )
)
demand_prob

In [ ]:
# TODO: Ensure Python >= 3.7 so order is preserved (matches insertion order)
demand_prob = {5: 1., 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0}
# Draw dictionary keys with corresponding probabilities
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)
torch.tensor(list(demand_prob.keys()))[index]


In [ ]:
demand_prob = {5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1}
# Draw dictionary keys with corresponding probabilities
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)
torch.tensor(list(demand_prob.keys()))[index]



# Use torch multinomial to sample from dictionary demand_prob
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)

torch.tensor(list(demand_prob.keys()))[index]

In [ ]:
demand_prob.values()

In [ ]:


demand_prob = {5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1}
custom_demand = CustomDemand(demand_prob)
custom_demand.sample(100, 3)
custom_demand.enumerate_support()

In [ ]:
controller.predict(current_inventory=2, past_regular_orders=[1, 1, 1])

In [ ]:
current_inventory = 10
past_regular_orders = [1, 2, 3, 4, 5]
past_expedited_orders = [6, 7, 8]
regular_lead_time = 3



In [ ]:
import numpy as np
current_inventory = 10
regular_lead_time = 5
expedited_lead_time = 2
past_regular_orders = np.array([1, 2, 3, 4, 5, 6, 7, 8])
past_expedited_orders = np.array([6, 7, 8])
# Output:
past_orders = [4, 5, 6, 7, 8]

# Pad with zeros for past_expedited_orders
# e_2 = np.pad(e_2, (0, 3), 'constant')
# Start with np.zeros before padding
e_1 = np.zeros(regular_lead_time)
e_1 = past_regular_orders[-regular_lead_time:]

e_2 = np.zeros(regular_lead_time)
e_2[:expedited_lead_time] = past_expedited_orders[-expedited_lead_time:]
e_3 = e_1 + e_2

first = current_inventory + e_3[0]
second = e_3[-regular_lead_time+1:]
print(tuple([first]+second))

In [ ]:
first =  past_regular_orders[-regular_lead_time]
second = past_regular_orders[-regular_lead_time+1:]
print(tuple([first]+second))

In [ ]:
controller = CappedDualIndexController()
controller.fit(sourcing_model, sourcing_periods=1000)
controller.predict(
    current_inventory,
    past_regular_orders,
    past_expedited_orders
)

In [ ]:
sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)

controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=10000,
    tolerance=1
)